# Utilities & Extractietools

Verzameling van losse scripts voor camerakalibratie en video-analyse. Met de JSON Extractie Tool kan je zware video's eenmalig analyseren en de gemaakte detecties (bounding boxes en posities) extraheren naar een handig JSON-bestand voor verdere batch-analyse.

> **Vereisten & Opmerkingen:**
> - **Google Colab:** De *Interactieve Bridge Tool* maakt gebruik van HTML/JavaScript-rendering en moet in Google Colab gerund worden om de interface correct te laden.
> - **Video bestanden:** `DJI_0809.MP4` (voor de interactieve kalibratie of/en voor detectie-extractie) en `Marathon.mp4` (voor de detectie-extractie).
> - **API:** Een geldige Roboflow API-key voor de YOLO-inferentie.

## 1. Video Specificaties Tool

In [ ]:
import cv2
from datetime import timedelta

VIDEO_PATH_EXT = '../../data/video/DJI_0809.MP4'

def print_video_specs(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Kan video niet openen: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps > 0:
        duration_sec = total_frames / fps
        duration_formatted = str(timedelta(seconds=int(duration_sec)))
    else:
        duration_sec = 0
        duration_formatted = "Onbekend"

    cap.release()

    print("--- VIDEO SPECIFICATIES ---")
    print(f"Bestand:    {video_path.split('/')[-1]}")
    print(f"Resolutie:  {width} x {height} pixels")
    print(f"Framerate:  {fps:.2f} fps")
    print(f"Totaal:     {total_frames} frames")
    print(f"Duur:       {duration_formatted} (ofwel {duration_sec:.2f} seconden)")

print_video_specs(VIDEO_PATH_EXT)

## 2. JSON Extractie Tool

In [ ]:
import json
import supervision as sv
from inference import get_model
from tqdm.notebook import tqdm

VIDEO_PATH_PUURS = '../../data/video/Marathon.mp4'
JSON_OUTPUT = '../../data/detecties/dataset_B_peloton.json'
ROBOFLOW_API_KEY = "API_KEY"
BEST_MODEL_ID = "tracking-runners/8"
CONFIDENCE_THRESHOLD = 0.55
LOST_TRACK_BUFFER = 30

model = get_model(model_id=BEST_MODEL_ID, api_key=ROBOFLOW_API_KEY)
tracker = sv.ByteTrack(lost_track_buffer=LOST_TRACK_BUFFER)
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH_PUURS)
cap = cv2.VideoCapture(VIDEO_PATH_PUURS)

export_data = {
    "fps": video_info.fps,
    "total_frames": video_info.total_frames,
    "tracks": {}
}

print(f"Start extractie van {video_info.total_frames} frames ({video_info.fps} fps)...")

for frame_idx in tqdm(range(video_info.total_frames), desc="Extractie"):
    success, frame = cap.read()
    if not success: break
        
    results = model.infer(frame)[0]
    detections = sv.Detections.from_inference(results)
    detections = detections[detections.confidence > CONFIDENCE_THRESHOLD]
    tracks = tracker.update_with_detections(detections=detections)
    
    if tracks.tracker_id is not None:
        for i, t_id in enumerate(tracks.tracker_id):
            t_id = int(t_id)
            bbox = tracks.xyxy[i]
            x_center = float((bbox[0] + bbox[2]) / 2.0)
            y_center = float((bbox[1] + bbox[3]) / 2.0)
            y_bottom = float(bbox[3])
            
            if str(t_id) not in export_data["tracks"]:
                export_data["tracks"][str(t_id)] = {"frames": [], "x_coords": [], "y_coords": [], "y_bottoms": []}
                
            export_data["tracks"][str(t_id)]["frames"].append(int(frame_idx))
            export_data["tracks"][str(t_id)]["x_coords"].append(x_center)
            export_data["tracks"][str(t_id)]["y_coords"].append(y_center)
            export_data["tracks"][str(t_id)]["y_bottoms"].append(y_bottom)

cap.release()

with open(JSON_OUTPUT, 'w') as f:
    json.dump(export_data, f)
print(f"\nExtractie voltooid! Opgeslagen in: {JSON_OUTPUT}")

## 3. Interactieve Keyframe Extractor (Klik op de brug)

In [ ]:
import base64
from IPython.display import display, HTML

INTERVAL_SECONDEN = 5
cap = cv2.VideoCapture(VIDEO_PATH_EXT)

if cap.isOpened():
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    interval_frames = int(fps * INTERVAL_SECONDEN)
    frame_indices = list(range(0, total_frames, interval_frames))
    
    images_data = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_resized = cv2.resize(frame, (1280, 720))
            _, buffer = cv2.imencode('.jpg', frame_resized, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
            b64 = base64.b64encode(buffer).decode('utf-8')
            images_data.append({"idx": idx, "b64": b64})
            
    cap.release()
    js_data = json.dumps(images_data)

    html_code = f"""
    <div id="app-container" style="font-family: Arial, sans-serif; max-width: 1280px; margin: auto; padding: 20px; background: #f9f9f9; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <h2 style="color: #333; margin-top: 0;">📍 Brug Kalibratie Tool</h2>
        <div id="instructions-container" style="display: flex; justify-content: space-between; align-items: center; background: #e0f2fe; padding: 15px; border-radius: 6px; margin-bottom: 15px;">
            <div style="font-size: 16px;">
                <strong style="color: #0284c7;">Huidig Frame:</strong> <span id="lbl-frame" style="font-weight: bold;">0</span> / {total_frames} <br>
                <strong style="color: #0284c7;">Volgende klik:</strong> <span id="lbl-instructie" style="font-weight: bold; color: #dc2626; font-size: 18px;">Top-Links</span>
            </div>
            <div>
                <button onclick="resetCurrentFrame()" style="background: #f59e0b; color: white; border: none; padding: 8px 15px; border-radius: 4px; cursor: pointer; font-weight: bold;">Reset dit frame</button>
            </div>
        </div>
        <div id="image-wrapper" style="position: relative; display: inline-block; cursor: crosshair;">
            <img id="main-image" style="width: 100%; height: auto; display: block; border-radius: 4px;" />
            <div id="dots-container" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none;"></div>
        </div>
        <div id="result-container" style="display: none; margin-top: 20px;">
            <h3 style="color: #16a34a;">✅ Kalibratie Voltooid! Kopieer onderstaande code:</h3>
            <textarea id="output-code" style="width: 100%; height: 250px; font-family: monospace; font-size: 14px; padding: 10px; border-radius: 4px; border: 1px solid #ccc;" readonly></textarea>
        </div>
    </div>

    <script>
        const origWidth = {orig_w};
        const origHeight = {orig_h};
        const imagesData = {js_data};
        let currentIndex = 0;
        let currentClicks = [];
        let allResults = [];
        const hoekpunten = ["Top-Links", "Top-Rechts", "Onder-Links", "Onder-Rechts"];
        const dotColors = ["#ef4444", "#3b82f6", "#eab308", "#22c55e"];
        const imgEl = document.getElementById('main-image');
        const dotsContainer = document.getElementById('dots-container');
        const lblFrame = document.getElementById('lbl-frame');
        const lblInstructie = document.getElementById('lbl-instructie');

        function loadImage(index) {{
            if(index >= imagesData.length) {{
                showResults();
                return;
            }}
            currentClicks = [];
            dotsContainer.innerHTML = '';
            lblFrame.innerText = imagesData[index].idx;
            updateInstruction();
            imgEl.src = "data:image/jpeg;base64," + imagesData[index].b64;
        }}

        function updateInstruction() {{
            lblInstructie.innerText = hoekpunten[currentClicks.length];
            lblInstructie.style.color = dotColors[currentClicks.length];
        }}

        function resetCurrentFrame() {{
            currentClicks = [];
            dotsContainer.innerHTML = '';
            updateInstruction();
        }}

        imgEl.addEventListener('mousedown', function(e) {{
            if(currentClicks.length >= 4) return;
            const rect = imgEl.getBoundingClientRect();
            const xClick = e.clientX - rect.left;
            const yClick = e.clientY - rect.top;
            
            const ratioX = origWidth / imgEl.clientWidth;
            const ratioY = origHeight / imgEl.clientHeight;
            const realX = Math.round(xClick * ratioX);
            const realY = Math.round(yClick * ratioY);
            
            currentClicks.push([realX, realY]);
            
            const dot = document.createElement('div');
            dot.style.position = 'absolute';
            dot.style.left = xClick + 'px';
            dot.style.top = yClick + 'px';
            dot.style.width = '12px';
            dot.style.height = '12px';
            dot.style.background = dotColors[currentClicks.length - 1];
            dot.style.borderRadius = '50%';
            dot.style.transform = 'translate(-50%, -50%)';
            dot.style.border = '2px solid white';
            dot.style.boxShadow = '0 0 4px rgba(0,0,0,0.5)';
            dotsContainer.appendChild(dot);
            
            if(currentClicks.length < 4) {{
                updateInstruction();
            }} else {{
                lblInstructie.innerText = "Opslaan...";
                lblInstructie.style.color = "#666";
                const frameIdx = imagesData[currentIndex].idx;
                const formatArray = [
                    currentClicks[0][1], currentClicks[1][1],
                    currentClicks[2][1], currentClicks[3][1]
                ];
                allResults.push("    " + frameIdx + ": [" + formatArray.join(", ") + "]");
                
                setTimeout(() => {{
                    currentIndex++;
                    loadImage(currentIndex);
                }}, 400);
            }}
        }});

        function showResults() {{
            document.getElementById('instructions-container').style.display = 'none';
            document.getElementById('image-wrapper').style.display = 'none';
            document.getElementById('result-container').style.display = 'block';
            let finalCode = "keyframes = {\n";
            finalCode += allResults.join(",\n");
            finalCode += "\n}";
            document.getElementById('output-code').value = finalCode;
        }}

        loadImage(0);
    </script>
    """
    display(HTML(html_code))